In [1]:
%%pyspark default.spark

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Yelp Sentiment Analysis") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

Executing for connection type: SPARK_GLUE, connection name: default.spark
Creating Glue session...
Create session for connection: default.spark


'Session 6ta2h2hm7s42zr-4b0e9c79-bd63-41b0-a6c4-e091c0e5d92b has been created.'

<sagemaker_studio_dataengineering_sessions.sagemaker_base_session_manager.common.debugging_utils.SessionInfoTableDisplay object>

Session created for connection: default.spark.


Connection: default.spark | Run start time: 2026-07-30 09:09:08.352790 | Run duration : 0:01:31.563240s.


In [2]:
%%pyspark default.spark
print(spark.version)


3.5.4-amzn-0

Connection: default.spark | Run start time: 2026-07-30 07:23:04.886289 | Run duration : 0:00:08.474406s.


In [5]:
%%pyspark default.spark

file_path = "s3://yelpdatasetvita/gold_layer/ml/sentiment_features/part-00000-c3f61153-e06b-4f01-88ea-69e9a133a765-c000.snappy.parquet"

df = spark.read.parquet(file_path)

df.printSchema()

df.show(10, truncate=False)

root
 |-- review_id: string (nullable = true)
 |-- stars: float (nullable = true)
 |-- review_text: string (nullable = true)
 |-- review_date: date (nullable = true)
 |-- genuity_score: double (nullable = true)
 |-- sentiment_label: string (nullable = true)
 |-- user_id: string (nullable = true)

+----------------------+-----+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [6]:
%%pyspark default.spark

from pyspark.sql.functions import col

df = df.select(
    col("review_text"),
    col("sentiment_label"),
    col("stars")
)

df.show(5)

+--------------------+---------------+-----+
|         review_text|sentiment_label|stars|
+--------------------+---------------+-----+
|OMG! This is the ...|       positive|  5.0|
|Great Yogurt Plac...|       positive|  5.0|
|As a loyal Gold P...|       negative|  2.0|
|I have been comin...|       positive|  5.0|
|12.05 - the day p...|       positive|  5.0|
+--------------------+---------------+-----+
only showing top 5 rows

Connection: default.spark | Run start time: 2026-07-30 09:18:09.643511 | Run duration : 0:00:10.152798s.


In [7]:
%%pyspark default.spark
df = df.na.drop(
    subset=[
        "review_text",
        "sentiment_label"
    ]
)



Connection: default.spark | Run start time: 2026-07-30 09:19:30.247388 | Run duration : 0:00:04.044172s.


In [8]:
%%pyspark default.spark
print(df.count())


6990280

Connection: default.spark | Run start time: 2026-07-30 09:20:20.080497 | Run duration : 0:00:28.209684s.


In [9]:
%%pyspark default.spark
from pyspark.sql.functions import lower, regexp_replace, trim

df = df.withColumn(
    "clean_text",
    lower(col("review_text"))
)

df = df.withColumn(
    "clean_text",
    regexp_replace("clean_text", r"http\\S+", "")
)

df = df.withColumn(
    "clean_text",
    regexp_replace("clean_text", "<[^>]*>", "")
)

df = df.withColumn(
    "clean_text",
    regexp_replace("clean_text", "[^a-zA-Z\\s]", "")
)

df = df.withColumn(
    "clean_text",
    trim(col("clean_text"))
)



Connection: default.spark | Run start time: 2026-07-30 09:21:14.662168 | Run duration : 0:00:04.019213s.


In [10]:
%%pyspark default.spark
df.select(
    "review_text",
    "clean_text"
).show(5, truncate=False)


+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [11]:
%%pyspark default.spark
from pyspark.ml.feature import StringIndexer

labelIndexer = StringIndexer(
    inputCol="sentiment_label",
    outputCol="label"
)

labelModel = labelIndexer.fit(df)

df = labelModel.transform(df)



Connection: default.spark | Run start time: 2026-07-30 09:22:31.424710 | Run duration : 0:00:18.105665s.


In [12]:
%%pyspark default.spark
from pyspark.ml.feature import Tokenizer

tokenizer = Tokenizer(
    inputCol="clean_text",
    outputCol="words"
)

df = tokenizer.transform(df)

df.select("clean_text", "words").show(5, truncate=False)


+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [13]:
%%pyspark default.spark
from pyspark.ml.feature import StopWordsRemover

remover = StopWordsRemover(
    inputCol="words",
    outputCol="filtered_words"
)

df = remover.transform(df)

df.select("filtered_words").show(5, truncate=False)


+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|filtered_words                                                                                                                                                                                                                                                                                                                

In [14]:
%%pyspark default.spark
from pyspark.ml.feature import HashingTF

hashingTF = HashingTF(
    inputCol="filtered_words",
    outputCol="rawFeatures",
    numFeatures=20000
)

df = hashingTF.transform(df)

df.select("rawFeatures").show(5, truncate=False)


+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|rawFeatures                                                                                                                                                                                                                                                                           

In [15]:
%%pyspark default.spark
from pyspark.ml.feature import IDF

idf = IDF(
    inputCol="rawFeatures",
    outputCol="features"
)

idfModel = idf.fit(df)

df = idfModel.transform(df)

df.select("features").show(5, truncate=False)


+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [16]:
%%pyspark default.spark
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

print("Training Records:", train_df.count())
print("Testing Records :", test_df.count())


Training Records: 5591894
Testing Records : 1398386

Connection: default.spark | Run start time: 2026-07-30 09:27:44.160731 | Run duration : 0:04:03.663031s.


In [17]:
%%pyspark default.spark
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=20
)

lr_model = lr.fit(train_df)



Connection: default.spark | Run start time: 2026-07-30 09:37:29.053659 | Run duration : 0:04:10.200066s.


In [18]:
%%pyspark default.spark
predictions = lr_model.transform(test_df)

predictions.select(
    "clean_text",
    "sentiment_label",
    "prediction"
).show(10, truncate=False)


+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [19]:
%%pyspark default.spark

from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

accuracy = evaluator.evaluate(predictions)

print("Accuracy =", accuracy)

Accuracy = 0.8694165988503889

Connection: default.spark | Run start time: 2026-07-30 09:44:04.360073 | Run duration : 0:01:53.184047s.


In [20]:
%%pyspark default.spark
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

print("Training Records:", train_df.count())
print("Testing Records :", test_df.count())


Training Records: 5591894
Testing Records : 1398386

Connection: default.spark | Run start time: 2026-07-30 09:52:31.421139 | Run duration : 0:03:09.767629s.


In [21]:
%%pyspark default.spark

from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=20,
    regParam=0.0
)

lr_model = lr.fit(train_df)



Connection: default.spark | Run start time: 2026-07-30 10:02:22.396065 | Run duration : 0:04:06.428180s.


In [22]:
%%pyspark default.spark
predictions = lr_model.transform(test_df)

predictions.select(
    "clean_text",
    "sentiment_label",
    "label",
    "prediction",
    "probability"
).show(10, truncate=False)


+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [23]:
%%pyspark default.spark
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

accuracy = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
).evaluate(predictions)

print("Accuracy:", accuracy)


Accuracy: 0.8694165988503889

Connection: default.spark | Run start time: 2026-07-30 10:09:43.041320 | Run duration : 0:01:51.452913s.


In [24]:
%%pyspark default.spark
f1 = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
).evaluate(predictions)

print("F1 Score:", f1)


F1 Score: 0.8547330475739313

Connection: default.spark | Run start time: 2026-07-30 10:13:03.901006 | Run duration : 0:01:15.834091s.


In [27]:
%%pyspark default.spark
lr_model.write().overwrite().save(
    "s3://yelpdatasetvita/gold_layer/sentiment_analysis_Model/"
)




Connection: default.spark | Run start time: 2026-07-30 10:53:35.610186 | Run duration : 0:00:09.678574s.


In [26]:
%%pyspark default.spark
predictions.write.mode("overwrite").parquet(
    "s3://yelpdatasetvita/gold_layer/sentiment_analysis_output/"
)




Connection: default.spark | Run start time: 2026-07-30 10:49:39.293574 | Run duration : 0:02:30.564219s.


In [ ]:
%%pyspark default.spark

